# `C05` — Partial fill and the resting residual

Scenario **C05** of the Track A certification ladder (card `OMIE-CERT-C05-1`, [`agile-plan.md`](../../docs/omiecid/agile-plan.md); canonical spec [`scenario-ladder.md`](../../docs/omiecid/scenario-ladder.md) §C5). Oracle kind: **exact-parametric**. **High-risk scenario** (independence rule 4): the K4 audit twin is written by a genuinely separate agent.

**Terms used below** (defined once in `scenario-ladder.md`, "What eager, jitted and assertion mean here"): *eager* runs every JAX operation immediately (`jax.disable_jit()`), so a human can inspect each step; *jitted* traces and compiles the whole run (`jax.jit`), the mode training uses; the two must agree. An *assertion* records a failure and the notebook carries on: every check is evaluated, and the verdict comes at the end. Every pass/fail assertion here is an **exact integer equality or order** on order ids, prices in ticks (0.01 EUR/MWh), quantities in ticks (0.1 MW) and the engine's stored timestamps `(sec, nsec)`; euro figures are shown, never asserted.

## 1. Purpose

**Purpose.** Most orders in a real session do not fill completely, so the handling of what is left
over is the ordinary case, not an edge case. Three things are asserted separately: the residual
**quantity**, the **price** it rests at (its own limit, not the price it just traded at), and its
**timestamp**. This is also the first scenario where conservation invariant I2 does real work.

**The user ruled on 2026-09-23 that the residual keeps its time priority**; `market-spec.md` §5.2
states it since v2.9, sourced to Continuous Trading Matching Algorithm §4.3 and §5.1.3: "Partial
execution is NOT a modification and does NOT assign a new timestamp." §5.2 states the rule separately
for the incoming residual and for a partially executed resting order, so the notebook quotes a settled
rule for both cases rather than an inference.

The ruling is asserted twice over. First on the stored field: the residual's timestamp is
**unchanged from the original submission**. Then — and this is the assertion that carries the weight
— behaviourally: after the partial fill, an interloper arrives at the residual's price, a second
aggressor is sent, and the residual must fill **ahead of** it. Timestamp equality alone could pass on
a field that the priority comparison never actually reads, which is exactly the kind of defect a
stored-value check invites.

The risk this guards against is concrete: the natural way to implement a resting residual is to
re-submit it, and a re-submission ordinarily takes a fresh timestamp. An implementation that took
that route would lose the residual's queue position on every partial fill, conserve every quantity,
print no wrong trade, and pass every other invariant in the battery. The interloper is what makes it
visible.

**The simulator does exactly this today.** On 2026-09-29 it was confirmed that when an incoming NON
order partially fills, its residual rests with a **new** timestamp (at least one tick later than its
original), because the matching loop cancels and re-dispatches it; a competing order sent in the same
second then jumps the queue. The fix, **`OMIE-NON-REMAINDER-TS-1`**, merged on 2026-09-29
(`231a5c9`), and `market-spec.md` v2.9 states the rule explicitly. The C5 card depends on it, and
the dependency is met: C5 is now expected to **pass**, and it is the certification that shows the fix
holds on randomised draws, not just on the story's own fixtures.

The ruling also has a bearing on Track B worth recording: an agent whose resting order is partially
filled does not go to the back of the queue, so partial fills cannot be a source of passive-fill
starvation.

*(Verbatim from `scenario-ladder.md` §C5.)*

**Certifies** (card): §5.1's `PARTIALLY_EXECUTED + resting in BOOK` branch; §6.2 item 4 (NON residual rests); §5.2 as amended by the 2026-09-23 ruling (decision 5).

**How the claim is pinned.** Every draw is four NON orders on one contract, in this order: a **resting** order $r$; an **aggressor** $a$ on the other side, crossing $r$; an **interloper** $i$ at the **residual's price, on the residual's side**; and a **take** $t$ on the opposite side, crossing that price and sized **exactly** to the residual. Two constructions:

- `C05.aggressor_residual` — case (i): $q_a > q_r$, so $a$ fills $r$ completely and its residual $q_a - q_r$ rests **at $a$'s own limit**; the interloper then joins $a$'s side at that limit.
- `C05.resting_residual` — case (ii): $q_a < q_r$, so $a$ is used up and $r$ is **partially executed**; its residual $q_r - q_a$ stays at $r$'s limit; the interloper joins $r$'s side at that limit.

The take must fill the residual (its owner's order id) and leave the interloper untouched, because the residual's timestamp is older. Per run the notebook asserts, separately:

1. **quantity** — the residual is exactly $q_a - q_r$ (i) or $q_r - q_a$ (ii), in quantity ticks;
2. **price** — it rests at its own limit ($p_a$ in (i), $p_r$ in (ii)), not at the price it just traded at;
3. **timestamp, stored** — the engine's stored `(sec, nsec)` of the residual is the same in every book from the moment it rests until the take arrives; in (ii) that includes the book **before** the partial fill, and in (i) it equals the stamp the same order receives when it enters and rests **without** trading (the no-cross control, section 4); and it is strictly earlier than the interloper's;
4. **timestamp, behavioural** — the exact counterparty sequence: the take trades with the residual's owner, for the whole residual, at the residual's limit; the interloper stays whole (domain review §17.2, D3-08: pin the exact counterparty sequence where two orders compete at one price);
5. **I2** — every order's quantity is accounted for, in integer ticks: filled + resting = declared.

**Why a same-second twin.** The defect C5 targets re-stamped the residual "at least one tick later than its original", so that "a competing order sent in the same second then jumps the queue" (`scenario-ladder.md` §C5). The generator puts the interloper one tape second after the partial fill. Whether that alone separates the defect depends on how the engine encodes its stamps, which the spec does not state (finding F4). So every draw also runs a **same-second twin**: the interloper's tape timestamp is set equal to that of the row that made the partial fill. §5.2 orders orders by entry, from one sequential queue, so the expected result does not change; a re-stamped residual loses the tie.

**K2 findings** (recorded here; none blocks the expectation):

- **F1 — spec version.** The card cites `market-spec.md` v2.12; the file is now v2.13. v2.13 changed §4.2.1 Rule 3 and §6.2 item 7 only (OQ-31). The passages quoted in section 2 are checked against the current file at run time.
- **F2 — the status name is not observable.** The card's "simulator status `PARTIALLY_FILLED`" is the engine's enum; the battery's `RunRecord` exposes no per-order status. The §5.1 branch `PARTIALLY_EXECUTED + resting in BOOK` is certified by its observable definition instead: after the partial-fill submission the order has traded (its fills are exact) **and** rests in the book with the exact residual. The enum value itself is not asserted.
- **F3 — what "original timestamp" means for an incoming order.** §5.2 assigns the stamp on entry, and an incoming order has no stored value before it rests. The notebook takes, as its original stamp, the stamp the same order gets in the **no-cross control**: the same tape with the resting order moved one tick off crossing, so the aggressor enters at the same point of the global queue and rests whole. This rests on §5.2: the stamp depends on the order's place in the entry queue (orders and capacity requests), not on trades. Whether the stored `sec` equals the tape's timestamp column is not stated (as in C03's F2); it is reported, not asserted.
- **F4 — the drawn tape's discriminating power is encoding-dependent**; hence the same-second twin above.
- **F5 — some draw sizes are not in `draw.params`.** In `C05.aggressor_residual` the interloper's size is only a float (`q_interloper_mw`) and the take's size is absent; in `C05.resting_residual` the take's size is absent. They are read from the tape rows and cross-checked against `params` where `params` has them.
- **F6 — one crossing level.** Each draw has a single crossing resting order, so "$\text{filled} = \sum$ crossing depth" is the one-level case $q_r$; multi-level walks are C06's.
- **F7 — I4 cannot see this.** I4 ranks by the engine's own stored stamps (`unobserved: queue_timestamp_correctness`), so a re-stamped residual passes I4. The counterparty sequence (item 4) and the stamp checks (item 3) are what certify §5.2 here.


## 2. The rule, quoted

The governing passages, quoted **verbatim** and checked against the document each names at run time (`notebooks.certification.cert_support.check_quotes`). A quote that has drifted is flagged **DRIFT** below and recorded as `quote_drift`: the notebook still runs to the end and ends `NOT CERTIFIED`. The regulatory sources behind §5.2 are shown as citations, not machine-checked.

How the quotes ground the expectation of section 5:

- **Which branch (§5.1).** An accepted order that trades on entry and still has quantity left is `PARTIALLY_EXECUTED + resting in BOOK` (case i); an order already `IN_BOOK` that trades part of its size is `PARTIALLY_EXECUTED (residual remains in book …)` (case ii). Only `MODIFIED` carries "→ new timestamp".
- **What rests, where (§4.5, §6.2 item 4).** NON allows a partial fill and a book entry; the incoming residual enters the book. It rests at its own limit: nothing in the spec changes an order's limit when it trades.
- **With which timestamp (§5.2, and the 2026-09-23 ruling).** A stamp is given on entry, from one global sequential queue. The new-timestamp list is modification, reactivation, a new iceberg slice; a partial execution is **not** on it, and §5.2 says so for both cases separately. The ruling is recorded in the domain review §8.
- **Who the take meets first (§6.1, §6.2 items 1–3).** Same price, so the earlier timestamp first: the residual, then the interloper. Each trade prints at the resting order's limit.


In [ ]:
import time

T0 = time.time()   # the measured wall clock (results: wall_clock_s) starts here

from IPython.display import Markdown, display

from notebooks.certification.cert_support import (
    Certification, Citation, SpecQuote, check_quotes, render_quotes,
)

QUOTES = (
    # --- §5.1: the two state-model branches C05 certifies -------------------------------------------
    SpecQuote(
        "market-spec.md", "§5.1",
        "└─ ACCEPTED\n         ├─ EXECUTED (fully matched immediately)\n"
        "         ├─ PARTIALLY_EXECUTED + resting in BOOK",
        note="Case (i): the incoming order trades on entry and its residual rests. The card's branch.",
    ),
    SpecQuote(
        "market-spec.md", "§5.1",
        "├─ PARTIALLY_EXECUTED (residual remains in book or cancelled per IOC)",
        note="Case (ii): an order already IN_BOOK is partially executed and its residual remains (NON).",
    ),
    SpecQuote(
        "market-spec.md", "§5.1",
        "├─ MODIFIED (price, quantity, or execution restriction change → new timestamp)",
        note="The only §5.1 branch that carries a new timestamp; a partial execution is not a modification.",
    ),
    # --- §5.2: the timestamp rules, the new-timestamp list and the ruling beside it -----------------
    SpecQuote(
        "market-spec.md", "§5.2",
        "Each order receives a timestamp when entering the SOB.",
        note="The stamp is given on entry. Finding F3: the no-cross control observes the entry stamp of case (i).",
    ),
    SpecQuote(
        "market-spec.md", "§5.2",
        "A **new timestamp** is assigned on: modification (price, quantity, execution restriction), "
        "reactivation from inactive, or activation of a new iceberg slice.",
        note="The new-timestamp list. A partial fill is not among the events that renew priority.",
    ),
    SpecQuote(
        "market-spec.md", "§5.2",
        "**Partial execution is NOT a modification and does NOT assign a new timestamp.** The unexecuted "
        "residual of a NON incoming order enters the book with its **original entry timestamp** (same order, "
        "not modified).",
        note="The 2026-09-23 ruling, case (i): asserted on C05.aggressor_residual.",
    ),
    SpecQuote(
        "market-spec.md", "§5.2",
        "Likewise a resting order that is partially executed keeps its timestamp.",
        note="The ruling, case (ii): asserted on C05.resting_residual.",
    ),
    SpecQuote(
        "market-spec.md", "§5.2",
        "Source: Continuous Trading Matching Algorithm §4.3 (the new-timestamp triggers are "
        "volume/price/execution-restriction *modification*, inactive→active, new iceberg slice) and §5.1.3 "
        "(\"the order is entered into the order book with its remaining quantity\" — the same order, only its "
        "quantity reduced by the trade per §5).",
        note="Provenance of the ruling in the regulatory corpus (shown as citations below).",
    ),
    SpecQuote(
        "market-spec.md", "§5.2",
        "All timestamps across orders and explicit capacity requests are assigned from a single global queue "
        "(unique, sequential). Source: Continuous Trading Matching Algorithm §4.3.",
        note="Stamps are unique and follow entry order: the residual (entered first) is strictly earlier than the "
             "interloper, also in the same-second twin.",
    ),
    SpecQuote(
        "scenario-ladder-review.md", "§8",
        "**Resolved — user ruling, 2026-09-23: the residual keeps its time priority.**",
        note="The user ruling (decision 5), as recorded in the domain review.",
    ),
    # --- §6.1 / §6.2: who the take meets, at what price ---------------------------------------------
    SpecQuote(
        "market-spec.md", "§6.1",
        "**Buy side:** ranked highest price first; ties broken by earliest timestamp.",
        note="Residual and interloper share one price: the earlier timestamp fills first (BUY-side residuals).",
    ),
    SpecQuote(
        "market-spec.md", "§6.1",
        "**Sell side:** ranked lowest price first; ties broken by earliest timestamp.",
        note="The mirror rule (SELL-side residuals).",
    ),
    SpecQuote(
        "market-spec.md", "§6.1",
        "A match occurs when best bid price ≥ best ask price.",
        note="The aggressor and the take each cross; the interloper meets an empty opposite side and rests.",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 1",
        "Incoming order attempts to match against best counterpart in the consolidated order book.",
        note="The take's best counterpart is the residual (same price, earlier stamp).",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 2",
        "Match is possible if: orders are opposite sides, same contract, and `sell_price ≤ buy_price`.",
        note="The crossing test used by the reference walk of section 5.",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 3",
        "Trade executes at the **resting order's limit price** (maker pricing).",
        note="The partial fill prints at r's limit; the take's fill prints at the residual's own limit.",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 4",
        "If incoming order is partially filled and has NON restriction: residual enters the book with its "
        "original entry timestamp (see §5.2).",
        note="Case (i): the aggressor's residual rests, keeping its entry timestamp.",
    ),
    # --- §4.5: NON rests --------------------------------------------------------------------------------
    SpecQuote(
        "market-spec.md", "§4.5",
        "| Restriction | Partial fill | Book entry if unmatched | Applicable to |",
        note="Header of the execution-restriction table, quoted so the NON row below can be read.",
    ),
    SpecQuote(
        "market-spec.md", "§4.5",
        "| NON | Yes | Yes | Regular, Block |",
        note="Every row of every C05 tape is NON: partial fills allowed, the remainder rests.",
    ),
)
CITATIONS = (
    Citation(
        "Continuous Trading Matching Algorithm (public description) §4.3 — "
        "docs/omiecid/corpus/Mapping_Authoritative_Documents_EU_Continuous_Intraday_Trading_SIDC_XBID_"
        "Spain_OMIE/Continuous_Trading_Matching_Algorithm_Public_Description.md",
        "When an order or an explicit capacity request is entered into the SOB (orders) or CMM (explicit "
        "requests), it receives a timestamp. An order receives a new timestamp whenever its volume, price or "
        "execution restriction is modified or its state is changed from inactive to active. Iceberg orders also "
        "receive a new time stamp whenever a new slice is activated. All timestamps are assigned sequentially",
    ),
    Citation(
        "Continuous Trading Matching Algorithm (public description) §5.1.3 — same file",
        "Subsequently , the order is deleted if the order quantity has reached zero or if it has the IOC "
        "execution restriction. In all other cases, the order is entered into the order book with its remaining "
        "quantity.",
    ),
)

QUOTE_CHECKS = check_quotes(QUOTES)
display(Markdown(render_quotes(QUOTE_CHECKS, CITATIONS)))


## 3. Configuration

One visible parameter cell. The headless gate overrides it by injecting a cell right after it (`execute_notebook(..., parameters={...})`); everything derived from it is computed in the next cell.

- **`DRAWS = 500`** valid draws of `C05.aggressor_residual` and **`RES_DRAWS = 500`** of `C05.resting_residual` (the card's default for a random scenario, per construction). The run requests `DRAWS + RES_DRAWS` valid draws.
- **`SEED0 = 0`** is the first seed of `C05.aggressor_residual`; **`RES_SEED0 = 1_000_000`** the first of `C05.resting_residual`. The two ranges are disjoint, so a seed names one draw only (in the results JSON and in a frozen counterexample). A draw found outside the capacity margin before the run, or marked I8 `invalid` by the battery after it, is **discarded and replaced by the next seed**. Each construction's loop stops at its draw count or after twice that many attempts, and the discard rate is reported.
- **`EAGER_DRAWS = 5`.** The first `EAGER_DRAWS` **valid** draws of each construction also run eager and are compared, integer field by integer field, with their jitted run (`compare_records`). On top of these, an eager run is **forced** for the first valid run of every class: construction × resting side × run kind (`draw`, `same_second`, and for `C05.aggressor_residual` also `no_cross_control`). Every one of these **must** have its eager comparison; a missing one is recorded as a failed `eager_jit` assertion. Eager runs are chosen among valid draws, so a discarded seed cannot remove them.
- The `same_second` twin runs on **every** valid draw, and the `no_cross_control` on every valid `C05.aggressor_residual` draw.
- The tick grids are **not** parameters: they come from `OMIEProductConfig()` and serve only to display euros. Every assertion is in integer ticks, order ids and stored `(sec, nsec)` pairs.

A run with far fewer draws than the default can miss a class (a resting side, or an interloper larger / not larger than the residual). It then fails the coverage check and the missing class's eager comparison. That is intended: such a run does not cover the card's dataset.


In [ ]:
CERT_ID = "C05"        # C00-C17 are the ladder; C90-C99 are reserved for the harness
DRAWS = 500            # valid draws of C05.aggressor_residual (card: random scenarios default to 500 draws)
RES_DRAWS = 500        # valid draws of C05.resting_residual (the card's second draw, case ii)
SEED0 = 0              # first seed of C05.aggressor_residual
RES_SEED0 = 1_000_000  # first seed of C05.resting_residual (a disjoint range: one seed names one draw)
EAGER_DRAWS = 5        # the first EAGER_DRAWS valid draws of each construction also run eager (plus forced classes)


In [ ]:
import pathlib

import numpy as np

from omiecid_simulator.jaxomiecid import omie_tape_synth as synth
from omiecid_simulator.omiecid.omie_config import OMIEProductConfig
from omiecid_simulator.omiecid.order_types import ExecRestriction, Side, Validity

AGGRESSOR = "C05.aggressor_residual"
RESTING = "C05.resting_residual"
CONSTRUCTIONS = (AGGRESSOR, RESTING)
N_DRAWS = {AGGRESSOR: DRAWS, RESTING: RES_DRAWS}
FIRST_SEED = {AGGRESSOR: SEED0, RESTING: RES_SEED0}
REQUESTED = DRAWS + RES_DRAWS
RUN_KINDS = {AGGRESSOR: ("draw", "same_second", "no_cross_control"), RESTING: ("draw", "same_second")}

CONFIG = OMIEProductConfig()
ENVELOPE = synth.DEFAULT_ENVELOPE
BUY, SELL = int(Side.BUY), int(Side.SELL)
NON, ICE = int(ExecRestriction.NON), int(ExecRestriction.ICE)
GFS = int(Validity.GFS)

SLICE_MARGIN = 64             # capacity-validity precondition: every iceberg has ceil(total/peak) <= 64
PRICE_INT_MAX = 1_999_800     # §2.3: +9,999 EUR/MWh in the offset encoding (0 is -9,999)
QTY_INT_MIN, QTY_INT_MAX = 1, 9_990   # §2.3: 0.1 MW and 999 MW in quantity ticks
TAKE_SUBMISSION = 3           # 0-based submission index of the take (the fourth real row)
PARTIAL_SUBMISSION = 1        # 0-based submission index of the aggressor (the partial fill)

_own = sorted(pathlib.Path.cwd().glob(f"{CERT_ID}_*.ipynb"))
NOTEBOOK = _own[0].name if len(_own) == 1 else "C05_partial_fill_residual.ipynb"

print(f"{CERT_ID}: {AGGRESSOR} x {DRAWS} (seeds from {SEED0}), {RESTING} x {RES_DRAWS} (seeds from {RES_SEED0}); "
      f"requested {REQUESTED} valid draws, at most 2x attempts per construction")
print(f"eager: first {EAGER_DRAWS} valid draws per construction + one forced run per "
      f"construction x resting side x run kind {dict(RUN_KINDS)}")
print(f"price tick {CONFIG.price_tick} EUR/MWh, qty tick {CONFIG.qty_tick} MW (display only)")
print(f"capacity envelope: {ENVELOPE}; slice margin <= {SLICE_MARGIN} per iceberg")


## 4. Dataset

**`C05.aggressor_residual`** (`omie_tape_synth.make_partial_fill_residual`, `variant="aggressor"`) and **`C05.resting_residual`** (`variant="resting"`) each emit four NON, GFS rows on one contract, with order ids $1, 2, 3, 4$ and tape timestamps $1, 2, 3, 4$ in row order. Write $\sigma$ for the resting side and $\bar\sigma$ for the other side.

| row | id | (i) `aggressor_residual` | (ii) `resting_residual` |
|---|---|---|---|
| $r$ resting | 1 | $\sigma$, $p_r$, $q_r$ | $\sigma$, $p_r$, $q_r$ |
| $a$ aggressor | 2 | $\bar\sigma$, $p_a$ crossing $p_r$, $q_a > q_r$ | $\bar\sigma$, $p_a$ crossing $p_r$, $q_a < q_r$ |
| $i$ interloper | 3 | $\bar\sigma$ at $p_a$ (the residual's price), $q_i$ | $\sigma$ at $p_r$ (the residual's price), $q_i$ |
| $t$ take | 4 | $\sigma$, limit crossing $p_a$, $q_t = q_a - q_r$ | $\bar\sigma$, limit crossing $p_r$, $q_t = q_r - q_a$ |

**What section 5 reads.** The expectation reads the **tape rows**: their order, sides, prices, quantities and order ids, through one reference walk; `draw.params` gives the closed form, cross-checked against the walk on every run, and is checked against the tape. `draw.expected` is never read.

**Derived runs**, made from every valid draw. Each changes only what its row says; neither is a separate draw.

| run | made from | change | purpose |
|---|---|---|---|
| `same_second` | every draw | the interloper's tape timestamp set equal to the aggressor's (the row that makes the partial fill) | separates a residual re-stamped "one tick later" whatever the engine's stamp encoding (finding F4) |
| `no_cross_control` | every (i) draw | the resting order $r$ moved one price tick off crossing: $p_a - 1$ if $\sigma$ = BUY, $p_a + 1$ if SELL | $a$ enters at the same point of the entry queue and rests **whole**: its stored stamp there is its original entry stamp (finding F3) |

**The precondition, checked from the tape for every valid draw** (an aggregate assertion in section 7): four real rows, all NON, GFS, no peak, one contract; order ids $1..4$ equal to `params`' `rest_oid`, `agg_oid`, `interloper_oid`, `take_oid`; tape timestamps strictly increasing; prices and quantities within §2.3's bounds; the sides, prices and sizes of the table above, with every `params` integer equal to the tape's (and, for (i), `encode_qty(q_interloper_mw)` equal to $q_i$); `params["residual_oid"]` the residual's owner. Each derived tape stays within §2.3's bounds.

**The capacity margin** (scenario-ladder-plan, *The capacity-validity precondition*, five ceilings). Four NON orders and no icebergs sit far inside it (at most 2 resting per side, at most 4 trade units, a hop bound of 5). It is still checked **before** each run on draws and derived tapes alike: a tape whose analytic bounds exceed the envelope (≤ 80 resting per side, ≤ 80 trades, hop bound ≤ 160, ≤ 800 concurrent icebergs), or whose largest $\lceil qty/peak \rceil$ exceeds 64, is never run. A draw is then discarded and re-drawn; a derived tape is recorded as a failed check. The slice count is re-checked from the tape and reported (0 here). After the run, the battery's I8 confirms the flags.

Below: the first seed of each construction in readable units beside the integers, with its derived tapes, the book shape, and the dataset's class counts.


In [ ]:
from collections import Counter

from omiecid_simulator.omiecid.encoding import decode_price, decode_qty, encode_qty

C_CONTRACT, C_SIDE, C_PRICE, C_QTY, C_OID = (synth.COL_CONTRACT, synth.COL_SIDE, synth.COL_PRICE, synth.COL_QTY,
                                             synth.COL_OID)
C_AGENT, C_TS, C_EXEC, C_VALIDITY, C_PEAK = (synth.COL_AGENT, synth.COL_TS, synth.COL_EXEC, synth.COL_VALIDITY,
                                             synth.COL_PEAK)
ROLE = ("r (resting)", "a (aggressor)", "i (interloper)", "t (take)")


def real_rows(tape):
    # The real rows (order_id >= 1), in tape (= submission) order.
    t = np.asarray(tape)
    return t[t[:, C_OID] >= 1]


def real_index(tape):
    # Tape row indices of the real rows, in order.
    return [int(i) for i in np.nonzero(np.asarray(tape)[:, C_OID] >= 1)[0]]


def side_name(side):
    return "BUY" if int(side) == BUY else "SELL"


def other(side):
    return SELL if int(side) == BUY else BUY


def crosses(rest_side, p_rest, p_in):
    # §6.1 / §6.2 item 2: sell price <= buy price, for a resting order on rest_side and an incoming limit p_in.
    return int(p_rest) <= int(p_in) if int(rest_side) == SELL else int(p_rest) >= int(p_in)


def max_slices(tape):
    # Largest ceil(qty / peak) over the tape's iceberg rows (0 when there are none).
    ice = [r for r in real_rows(tape) if int(r[C_EXEC]) == ICE]
    return max((-(-int(r[C_QTY]) // max(1, int(r[C_PEAK]))) for r in ice), default=0)


def same_second_tape(tape):
    # The interloper (third real row) takes the tape timestamp of the aggressor (second real row).
    t = np.array(tape, dtype=np.int32, copy=True)
    idx = real_index(t)
    t[idx[2], C_TS] = t[idx[1], C_TS]
    return np.ascontiguousarray(t)


def control_tape(tape):
    # The resting order moved one tick off crossing the aggressor; nothing else changes.
    t = np.array(tape, dtype=np.int32, copy=True)
    idx = real_index(t)
    rest_side, p_in = int(t[idx[0], C_SIDE]), int(t[idx[1], C_PRICE])
    t[idx[0], C_PRICE] = p_in - 1 if rest_side == BUY else p_in + 1
    return np.ascontiguousarray(t)


DRAWN, DERIVED = {}, {}


def prepare(key, seed):
    # Draw one seed of one construction and build its derived tapes.
    d = synth.CONSTRUCTIONS[key](seed)
    DRAWN[(key, seed)] = d
    DERIVED[(key, seed, "draw")] = np.asarray(d.tape)
    DERIVED[(key, seed, "same_second")] = same_second_tape(d.tape)
    if key == AGGRESSOR:
        DERIVED[(key, seed, "no_cross_control")] = control_tape(d.tape)
    return d


for _key in CONSTRUCTIONS:
    for _s in range(FIRST_SEED[_key], FIRST_SEED[_key] + N_DRAWS[_key]):
        prepare(_key, _s)


def tape_table(tape, title):
    rows = ["| role | oid | side | price (EUR/MWh) | qty (MW) | price_int | qty_int | agent | tape ts | exec |",
            "|---|---|---|---|---|---|---|---|---|---|"]
    for k, r in enumerate(real_rows(tape)):
        rows.append(f"| {ROLE[k] if k < 4 else '?'} | {int(r[C_OID])} | {side_name(r[C_SIDE])} | "
                    f"{decode_price(int(r[C_PRICE]), CONFIG):.2f} | {decode_qty(int(r[C_QTY]), CONFIG):.1f} | "
                    f"{int(r[C_PRICE])} | {int(r[C_QTY])} | {int(r[C_AGENT])} | {int(r[C_TS])} | "
                    f"{'NON' if int(r[C_EXEC]) == NON else int(r[C_EXEC])} |")
    return f"**{title}**\n\n" + "\n".join(rows)


for _key in CONSTRUCTIONS:
    _s = FIRST_SEED[_key]
    _d = DRAWN[(_key, _s)]
    parts = [tape_table(_d.tape, f"{_key}, seed {_s} (digest `{_d.digest()[:16]}…`)")]
    for _kind in RUN_KINDS[_key][1:]:
        parts.append(tape_table(DERIVED[(_key, _s, _kind)], f"{_key}, seed {_s}: `{_kind}`"))
    display(Markdown("\n\n".join(parts)))
    print(f"{_key} seed {_s} params: {_d.params}")


In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(11, 2.8))
for ax, _key in zip(axes, CONSTRUCTIONS):
    _d = DRAWN[(_key, FIRST_SEED[_key])]
    for k, r in enumerate(real_rows(_d.tape)):
        buy = int(r[C_SIDE]) == BUY
        q = decode_qty(int(r[C_QTY]), CONFIG)
        p = decode_price(int(r[C_PRICE]), CONFIG)
        ax.barh(p + 0.08 * (k - 1.5), q * (1 if buy else -1), height=0.07,
                color="tab:green" if buy else "tab:red")
        ax.annotate(f"{ROLE[k].split()[0]} (ts {int(r[C_TS])})", (q * (1 if buy else -1), p + 0.08 * (k - 1.5)),
                    fontsize=7, va="center")
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("quantity (MW): buy right, sell left")
    ax.set_ylabel("limit (EUR/MWh)")
    ax.set_title(f"{_key}, seed {FIRST_SEED[_key]}: rows in entry order", fontsize=9)
plt.tight_layout()
plt.show()

_classes = Counter()
for (_key, _s), _d in DRAWN.items():
    _r = real_rows(_d.tape)
    _res = (int(_r[1][C_QTY]) - int(_r[0][C_QTY])) if _key == AGGRESSOR else (int(_r[0][C_QTY]) - int(_r[1][C_QTY]))
    _classes[(_key, f"resting {side_name(_r[0][C_SIDE])}")] += 1
    _classes[(_key, "q_i > residual" if int(_r[2][C_QTY]) > _res else "q_i <= residual")] += 1
display(Markdown("**Classes in the initial seed range** (counts)\n\n| construction | class | draws |\n|---|---|---|\n"
                 + "\n".join(f"| {k} | {c} | {n} |" for (k, c), n in sorted(_classes.items()))))


## 5. Known result

**Notation.** Integer ticks throughout. The tape's four real rows, in submission order $s = 0, 1, 2, 3$, are $r, a, i, t$ (section 4). Every row is NON on one contract (section 4's precondition).

**(W) The reference walk** (§6.1, §6.2 items 1–4, §4.5, §5.2). Process the rows in submission order. An entering order with limit $p$ and size $q$ meets the opposite side ranked by $(p_e, k_e)$ (SELL resting: lowest price first) or $(-p_e, k_e)$ (BUY resting: highest price first), where $k_e$ is the entry's **priority key**. While it has quantity left and the best opposite entry crosses (§6.2 item 2), it trades $f = \min(\text{left}, q_e)$ **at $p_e$** (§6.2 item 3). An entry used up leaves the book. Whatever is left of the incoming order rests at **its own limit** $p$ (§4.5 NON; §6.2 item 4).

The priority key is the order's **entry index** $s$, given once when the order enters (§5.2: one global sequential queue) and **never changed by a partial execution** (§5.2, both cases; the 2026-09-23 ruling). This is the rule. Beside it, a **re-stamp model** gives a partially executed order (incoming residual or resting order) the key $s + 1.5$, one entry later than the submission that partially filled it: the defect C5 targets. Section 7 checks that the dataset separates the two models on every run.

**Closed form** (from `draw.params`, with $q_i$ and $q_t$ read from the tape rows, finding F5). Case (i), $R = q_a - q_r$:

| | fills `[passive, aggressor, price_int, qty_int, submission]` | book after $s=1$ | book at the end |
|---|---|---|---|
| draw, `same_second` | $[r, a, p_r, q_r, 1]$, $[a, t, p_a, R, 3]$ | $\bar\sigma$: $[p_a, R, a]$ | $\bar\sigma$: $[p_a, q_i, i]$ |
| `no_cross_control` ($p_c = p_a \mp 1$) | $[a, t, p_a, R, 3]$ | $\sigma$: $[p_c, q_r, r]$; $\bar\sigma$: $[p_a, q_a, a]$ | $\sigma$: $[p_c, q_r, r]$; $\bar\sigma$: $[p_a, q_r, a], [p_a, q_i, i]$ |

Case (ii), $R = q_r - q_a$:

| | fills | book after $s=1$ | book at the end |
|---|---|---|---|
| draw, `same_second` | $[r, a, p_r, q_a, 1]$, $[r, t, p_r, R, 3]$ | $\sigma$: $[p_r, R, r]$ | $\sigma$: $[p_r, q_i, i]$ |

**What is asserted, per run** (draw and `same_second`; the residual's owner is $a$ in (i), $r$ in (ii)):

- **(A) acceptance** — all four orders accepted, none rejected.
- **(Q) counterparty sequence** — the exact fill list above, in trade-row order.
- **(F) the partial fill** — the quantity traded at $s = 1$ is $\sum$ crossing depth $= q_r$ in (i) and $q_a$ in (ii).
- **(R) the residual, quantity and price, separately** — in the book after $s = 1$, the owner's visible quantity is exactly $R$, and its only resting price is its own limit ($p_a$ in (i), $p_r$ in (ii)). With (Q) this is §5.1's `PARTIALLY_EXECUTED + resting in BOOK` (finding F2).
- **(B1) / (B) the book** after $s = 1$ and at the end, `[price_int, qty_int, oid]` per side.
- **(T1) stored timestamp unchanged** — the owner's stored `(sec, nsec)` read from every book between its first rest and the take (after $s = 0$ in (ii), after $s = 1$ in (i); then after $s = 2$, and just before $s = 3$): exactly one entry in each, all equal.
- **(T2) original entry stamp, case (i)** — the residual's stamp after $s = 1$ equals $a$'s stamp after $s = 1$ in the `no_cross_control`, where $a$ rests whole (finding F3).
- **(T3) older than the interloper** — just before the take, the residual's stamp is strictly earlier, lexicographically on `(sec, nsec)`, than the interloper's (§5.2, unique and sequential).
- **(O) fill order against the interloper** — the take's fills have exactly the residual's owner as passive side, and the interloper's filled quantity is 0.
- **(I2) conservation** — for every order, filled + resting at the end = declared size, in quantity ticks; and the battery's I2 status is `pass` (draw runs; for derived runs the whole battery is one assertion).

On the `no_cross_control`: (A), (Q), (B1), (B) and the battery. Its stamp feeds (T2).

The code below implements (W) as one function of the tape rows, used unchanged for every run and both models. It does not read `draw.expected` and it does not run the engine.


In [ ]:
def book_view(book):
    # {side: sorted [price_int, qty_int, oid]} of the walk's resting entries.
    out = {"BUY": [], "SELL": []}
    for e in book:
        out[side_name(e["side"])].append([e["price"], e["qty"], e["oid"]])
    return {k: sorted(v) for k, v in out.items()}


def walk(tape, model="rule"):
    # (W): the reference walk over the real rows. model="rule" keeps the entry key on a partial execution
    # (§5.2); model="restamp" moves a partially executed order to key s + 1.5 (the targeted defect).
    book, fills, books = [], [], []
    for s, r in enumerate(real_rows(tape)):
        side, p, q, oid = int(r[C_SIDE]), int(r[C_PRICE]), int(r[C_QTY]), int(r[C_OID])
        left = q
        while left > 0:
            opp = [e for e in book if e["side"] != side and crosses(e["side"], e["price"], p)]
            if not opp:
                break
            best = min(opp, key=lambda e: ((e["price"] if e["side"] == SELL else -e["price"]), e["key"]))
            f = min(left, best["qty"])
            fills.append([best["oid"], oid, best["price"], f, s])
            best["qty"] -= f
            left -= f
            if best["qty"] == 0:
                book.remove(best)
            elif model == "restamp":
                best["key"] = s + 1.5
        if left > 0:
            key = float(s) if (model == "rule" or left == q) else s + 1.5
            book.append({"side": side, "price": p, "qty": left, "oid": oid, "key": key})
        books.append(book_view(book))
    return {"fills": fills, "books": books}


def expectation(tape):
    # Everything section 7 compares, from (W) on the tape rows. None when the tape is not a 4-row C05 tape.
    rows = real_rows(tape)
    if len(rows) != 4:
        return None
    w = walk(tape)
    oids = [int(r[C_OID]) for r in rows]
    declared = {int(r[C_OID]): int(r[C_QTY]) for r in rows}
    sides = {int(r[C_OID]): int(r[C_SIDE]) for r in rows}
    fills = w["fills"]
    b1 = w["books"][PARTIAL_SUBMISSION]
    traded_at_1 = {o: sum(f[3] for f in fills if f[4] == PARTIAL_SUBMISSION and o in (f[0], f[1])) for o in oids}
    resting_1 = {o: sum(x[1] for v in b1.values() for x in v if x[2] == o) for o in oids}
    owners = [o for o in oids if traded_at_1[o] > 0 and resting_1[o] > 0]
    if len(owners) != 1:
        return None
    res = owners[0]
    res_side = sides[res]
    res_prices = sorted({x[0] for x in b1[side_name(res_side)] if x[2] == res})
    final = w["books"][-1]
    final_rest = {o: sum(x[1] for v in final.values() for x in v if x[2] == o) for o in oids}
    filled = {o: sum(f[3] for f in fills if o in (f[0], f[1])) for o in oids}
    return {
        "oids": oids, "fills": fills, "book_after_partial": b1, "book_final": final,
        "partial_fill_qty": sum(f[3] for f in fills if f[4] == PARTIAL_SUBMISSION),
        "res_oid": res, "res_side": res_side, "res_entry": oids.index(res),
        "res_qty": resting_1[res], "res_prices": res_prices,
        "interloper_oid": oids[2], "interloper_side": sides[oids[2]], "take_oid": oids[3],
        "take_passives": [f[0] for f in fills if f[4] == TAKE_SUBMISSION],
        "filled": filled, "declared": declared, "final_rest": final_rest,
    }


def closed_form(key, kind, p, tape):
    # The closed form of the tables above, from draw.params (q_i, q_t from the tape rows: finding F5).
    rows = real_rows(tape)
    q_i = int(rows[2][C_QTY])
    rs, as_ = side_name(p["rest_side"]), side_name(p["agg_side"])
    ro, ao, io, to = p["rest_oid"], p["agg_oid"], p["interloper_oid"], p["take_oid"]
    pr, pa, qr, qa = p["p_rest_int"], p["p_agg_int"], p["q_rest_int"], p["q_agg_int"]
    if key == AGGRESSOR:
        R = qa - qr
        if kind == "no_cross_control":
            pc = pa - 1 if p["rest_side"] == BUY else pa + 1
            return {"fills": [[ao, to, pa, R, 3]],
                    "book_after_partial": {rs: [[pc, qr, ro]], as_: [[pa, qa, ao]]},
                    "book_final": {rs: [[pc, qr, ro]], as_: sorted([[pa, qa - R, ao], [pa, q_i, io]])}}
        return {"fills": [[ro, ao, pr, qr, 1], [ao, to, pa, R, 3]],
                "book_after_partial": {rs: [], as_: [[pa, R, ao]]},
                "book_final": {rs: [], as_: [[pa, q_i, io]]},
                "res_oid": ao, "res_qty": R, "res_prices": [pa], "partial_fill_qty": qr}
    R = qr - qa
    return {"fills": [[ro, ao, pr, qa, 1], [ro, to, pr, R, 3]],
            "book_after_partial": {rs: [[pr, R, ro]], as_: []},
            "book_final": {rs: [[pr, q_i, io]], as_: []},
            "res_oid": ro, "res_qty": R, "res_prices": [pr], "partial_fill_qty": qa}


def closed_form_agrees(key, kind, draw, tape):
    cf = closed_form(key, kind, draw.params, tape)
    if kind == "no_cross_control":
        w = walk(tape)
        return (w["fills"] == cf["fills"] and w["books"][PARTIAL_SUBMISSION] == cf["book_after_partial"]
                and w["books"][-1] == cf["book_final"])
    e = expectation(tape)
    return e is not None and all(e[k] == cf[k] for k in cf)


def separates(tape):
    # The dataset tells the rule from the re-stamp model on this tape.
    a, b = walk(tape, "rule"), walk(tape, "restamp")
    return a["fills"] != b["fills"] or a["books"] != b["books"]


def precondition(key, draw):
    # Section 4's dataset precondition, from the tape and params. Returns the list of departures.
    bad, t, p = [], draw.tape, draw.params
    rows = real_rows(t)
    if len(rows) != 4:
        return [f"{len(rows)} real rows (expected 4)"]
    r, a, i, tk = rows
    rest_side = int(p.get("rest_side", -1))
    if any(int(x[C_EXEC]) != NON for x in rows):
        bad.append("a row is not NON")
    if any(int(x[C_VALIDITY]) != GFS for x in rows) or any(int(x[C_PEAK]) != 0 for x in rows):
        bad.append("a row is not GFS or carries a peak")
    if len({int(x[C_CONTRACT]) for x in rows}) != 1:
        bad.append("more than one contract")
    oids = [int(x[C_OID]) for x in rows]
    if oids != [1, 2, 3, 4] or oids != [p.get("rest_oid"), p.get("agg_oid"), p.get("interloper_oid"),
                                         p.get("take_oid")]:
        bad.append(f"order ids {oids} vs params")
    ts = [int(x[C_TS]) for x in rows]
    if any(b_ <= a_ for a_, b_ in zip(ts[:-1], ts[1:])):
        bad.append(f"tape timestamps not strictly increasing: {ts}")
    if any(not 0 <= int(x[C_PRICE]) <= PRICE_INT_MAX or not QTY_INT_MIN <= int(x[C_QTY]) <= QTY_INT_MAX
           for x in rows):
        bad.append("price or quantity outside §2.3")
    if int(r[C_SIDE]) != rest_side or int(a[C_SIDE]) != other(rest_side) or int(p.get("agg_side", -1)) != other(rest_side):
        bad.append("r / a sides")
    for name, x, col in (("p_rest_int", r, C_PRICE), ("q_rest_int", r, C_QTY), ("p_agg_int", a, C_PRICE),
                         ("q_agg_int", a, C_QTY)):
        if int(p.get(name, -1)) != int(x[col]):
            bad.append(f"params {name} != tape")
    pr, qr, pa, qa = int(r[C_PRICE]), int(r[C_QTY]), int(a[C_PRICE]), int(a[C_QTY])
    if not crosses(rest_side, pr, pa):
        bad.append("a does not cross r")
    if key == AGGRESSOR:
        R = qa - qr
        if not qa > qr:
            bad.append("q_a <= q_r")
        if int(i[C_SIDE]) != other(rest_side) or int(i[C_PRICE]) != pa:
            bad.append("interloper not on a's side at p_a")
        if int(tk[C_SIDE]) != rest_side or not crosses(other(rest_side), pa, int(tk[C_PRICE])):
            bad.append("take not on r's side crossing p_a")
        if "q_interloper_mw" in p and int(encode_qty(float(p["q_interloper_mw"]), CONFIG)) != int(i[C_QTY]):
            bad.append("params q_interloper_mw != tape")
        if int(p.get("residual_oid", -1)) != int(a[C_OID]):
            bad.append("params residual_oid is not a")
        ctrl = real_rows(DERIVED.get((key, draw.seed, "no_cross_control"), control_tape(t)))
        if not 0 <= int(ctrl[0][C_PRICE]) <= PRICE_INT_MAX or crosses(rest_side, int(ctrl[0][C_PRICE]), pa):
            bad.append("no_cross_control: r outside §2.3 or still crossing")
    else:
        R = qr - qa
        if not qa < qr:
            bad.append("q_a >= q_r")
        if int(i[C_SIDE]) != rest_side or int(i[C_PRICE]) != pr:
            bad.append("interloper not on r's side at p_r")
        if int(tk[C_SIDE]) != other(rest_side) or not crosses(rest_side, pr, int(tk[C_PRICE])):
            bad.append("take not on a's side crossing p_r")
        for name, x, col in (("q_interloper_int", i, C_QTY), ("p_take_int", tk, C_PRICE)):
            if name in p and int(p[name]) != int(x[col]):
                bad.append(f"params {name} != tape")
        if int(p.get("residual_oid", -1)) != int(r[C_OID]):
            bad.append("params residual_oid is not r")
    if int(tk[C_QTY]) != R:
        bad.append(f"take size {int(tk[C_QTY])} != residual {R}")
    if max_slices(t) != 0:
        bad.append("an iceberg row")
    return bad


# ---- the expectation on the first seed of each construction, every run kind ------------------------------
CF_AGREES = True
for _key in CONSTRUCTIONS:
    _s = FIRST_SEED[_key]
    _d = DRAWN[(_key, _s)]
    for _kind in RUN_KINDS[_key]:
        _t = DERIVED[(_key, _s, _kind)]
        _w = walk(_t)
        _ok = closed_form_agrees(_key, _kind, _d, _t)
        CF_AGREES &= _ok
        print(f"{_key} seed {_s} {_kind}: fills {_w['fills']}")
        print(f"    book after s=1 {_w['books'][PARTIAL_SUBMISSION]}; at the end {_w['books'][-1]}")
        print(f"    closed form == walk: {_ok}; re-stamp model fills {walk(_t, 'restamp')['fills']} "
              f"(separates: {separates(_t)})")
    _e = expectation(DERIVED[(_key, _s, 'draw')])
    print(f"    residual: owner {_e['res_oid']} on {side_name(_e['res_side'])}, qty_int {_e['res_qty']} "
          f"({decode_qty(_e['res_qty'], CONFIG):.1f} MW) at {_e['res_prices']} "
          f"({decode_price(_e['res_prices'][0], CONFIG):.2f} EUR/MWh); precondition departures: "
          f"{precondition(_key, _d)}")
print(f"closed form agrees with (W) on the first seeds: {CF_AGREES}")


## 6. Run

### Eager and jitted (wired at K3)

> **K3: this section is yours; sections 1–5 and 7 are locked (independence rule 2).** A disagreement with the expectation is escalated as a `SCOPE-CHANGE` candidate with both readings, never resolved by editing sections 5 or 7.

Section 7 consumes exactly **one name** from this section. Until it is callable, section 7 records a failing `section 6 wired` check and the notebook ends `NOT CERTIFIED`.

**`run_tape(tape, *, jit) -> RunRecord`**: the battery's record builder on one tape, i.e. `tests.omiecid.reference.omie_invariants.build_record(tape, jit=jit)`. It runs on one fresh book of an OPEN contract with the default configs (`OMIEProductConfig()`, `OMIEMarketConfig()`), no cancellations (`cancels_before={}`), no gate-closure expiry, and the default capacity bounds (`tape_capacity_bounds(tape)`, which equal `draw.capacity_bounds` on these draws). `jit=False` runs eager (the builder wraps the run in `jax.disable_jit()`); `jit=True` runs jitted. The tape is passed **as is**: `DERIVED[(key, seed, kind)]` for `kind` in `draw`, `same_second`, `no_cross_control` (the first is `draw.tape` itself). Nothing in the tape may be rewritten, and that includes its timestamp column.

Section 7 reads these `RunRecord` fields:
- `fills`: each `Fill`'s `passive_oid`, `aggressor_oid`, `price_int`, `qty_int`, `submission` and `row_index`;
- `accepted` (its keys), `rejections` (each `Rejection.oid`) and `n_submissions`;
- `post_snapshots[s]` for $s = 0, 1, 2$, `pre_snapshots[3]` and `final_snapshot`: `.resting(side)` (price, qty, oid, agent, sec, nsec) and `.visible_qty(oid, side)`;
- `filled`, `observed_capacity`;
- the whole record, through `run_battery` and `compare_records`.

Section 7 calls `run_tape(..., jit=True)` on every draw and every derived run, and `run_tape(..., jit=False)` on the eager selection of section 3. A seed beyond the initial range (a re-draw after a discard) is prepared with `prepare(key, seed)` before the call.


In [ ]:
# ---------------------------------------------------------------------------------------
# Section 6 (K3): run_tape(tape, *, jit) -> RunRecord, per the contract above.
#
# The battery's record builder on one fresh book of one OPEN contract, with the default
# configs (OMIEProductConfig(), OMIEMarketConfig()), no scheduled cancellations, no
# gate-closure expiry and the default analytic capacity bounds (tape_capacity_bounds(tape),
# equal to draw.capacity_bounds on these draws). jit=False wraps the whole run in
# jax.disable_jit() (inside build_record); jit=True runs it jitted. The tape is passed as
# is (DERIVED[(key, seed, kind)] for draw, same_second, no_cross_control): no column,
# including the timestamp column, is rewritten.
# ---------------------------------------------------------------------------------------
from omiecid_simulator.omiecid.contract import ContractStatus
from omiecid_simulator.omiecid.omie_config import OMIEMarketConfig
from tests.omiecid.reference.omie_invariants import build_record

MARKET = OMIEMarketConfig()


def run_tape(tape, *, jit):
    """One run of `tape` (a draw's tape, its same_second twin or its no_cross_control, unmodified) on a fresh
    OPEN book."""
    return build_record(tape, product_config=CONFIG, market_config=MARKET,
                        contract_status=ContractStatus.OPEN, cancels_before={},
                        expire_at_gct=False, capacity_bounds=None, jit=jit)


print(f"run_tape: build_record on one OPEN contract, {type(CONFIG).__name__}() + {type(MARKET).__name__}(), "
      f"no cancels, no GCT expiry, default capacity bounds")

## 7. Certification

The oracle comparison comes first, by order id, in integer ticks and on stored `(sec, nsec)` pairs. Then come the invariant battery (RB, I1–I9) as a pass/fail table, the capacity peaks and the discard rate. No check stops the notebook: every assertion is evaluated and recorded.

**What is recorded, per run** (`draw` and `same_second`), each with its expected and observed value **in full**, so an auditor can check every integer from the results JSON alone: (A) accepted order ids and rejections; (Q) the counterparty sequence; (F) the quantity traded at the partial fill; (R) the residual's quantity and, separately, its resting price; (B1) the book after the partial fill; (T1) the owner's stamps in every book of the window, with the snapshot each was read from; (T2, case i) the residual's stamp and the control's; (T3) the residual's and the interloper's stamps before the take; (O) the take's passive ids and the interloper's filled quantity; (B) the final book; (I2) filled + resting per order against the declared sizes, and, on draws, the battery's I2 status. **Per `no_cross_control`:** (A), (Q), (B1), (B) and its battery.

**Aggregate assertions:** the section-6 wiring; the dataset precondition per construction (section 4), listing every departing seed; the closed form against (W) on every run; every derived tape within §2.3 and the capacity margin; **separation** — the re-stamp model disagrees with (W) on every `draw` and `same_second` run; **coverage** — per construction both resting sides, and an interloper larger than the residual and not larger; and the eager comparisons, one per required class, a missing one recorded as a failure.

The battery runs on every draw's jitted record with `expects_trade=True`; a draw it marks I8 `invalid` is discarded and replaced by the next seed. For derived runs the battery is folded into one assertion each, since a derived run is not a draw. **Reported, not asserted** (finding F3): whether the stored `sec` of the residual equals its row's tape timestamp, and the stamps' `nsec` values.


In [ ]:
import traceback
from collections import defaultdict

from tests.omiecid.reference.omie_invariants import compare_records, run_battery

cert = Certification(
    cert_id=CERT_ID, notebook=NOTEBOOK, scenario="Partial fill and the resting residual",
    known_result="(i) filled = crossing depth q_r; the residual q_a - q_r rests at the aggressor's own limit p_a with "
                 "its original entry timestamp (equal to its stamp when it rests unmatched, unchanged until the take, "
                 "earlier than the interloper's); (ii) the resting order filled q_a keeps q_r - q_a at p_r with its "
                 "stamp unchanged; in both, the take fills the residual's owner for the whole residual at its limit "
                 "and the interloper stays whole, also when the interloper shares the tape second; I2 closes",
    oracle_kind="exact-parametric", quotes=QUOTES, citations=CITATIONS, constructions=CONSTRUCTIONS,
    requested_draws=REQUESTED, runtime_forecast_s=60.0, started_at=T0,
    downstream="every scenario where an order rests after trading: C06 (the last level's partial), C07 (the NON "
               "counterfactual of an IOC), C08-C10 (iceberg partials), C15 (random sessions), C17; Track B "
               "passive-fill attribution (a partially filled agent order keeps its queue position)",
)

R_ACC = "§5.1 SUBMITTED -> ACCEPTED; §2.3"
R_SEQ = "§6.1; §6.2 items 1-3 (rank order, maker price); §5.2 (no new timestamp on partial execution)"
R_PART = "§5.1 PARTIALLY_EXECUTED + resting in BOOK; §6.2 item 4; §4.5 NON"
R_BOOK = "§4.5 NON; §6.2 item 4"
R_TS = "§5.2 (partial execution does NOT assign a new timestamp; 2026-09-23 ruling)"
R_TS_ENTRY = "§5.2 (timestamp on entry; original entry timestamp); §6.2 item 4; finding F3"
R_TS_ORDER = "§5.2 (single global queue: unique, sequential); §6.1 (ties by earliest timestamp)"
R_I2 = "I2 (quantity conservation)"
R_DATA = "dataset precondition (section 4); §2.3"


def guarded(label, seed, fn, *args, **kw):
    # Call a section-6 function; a raise is recorded as a failed check, never propagated.
    try:
        return fn(*args, **kw)
    except Exception as exc:  # noqa: BLE001
        cert.check(f"{label}: run raised", False, seed=seed, expected="a RunRecord", observed=type(exc).__name__,
                   detail=f"{type(exc).__name__}: {exc} | " + traceback.format_exc().splitlines()[-1], rule=R_ACC)
        return None


def snap(rec, which, i):
    seq = rec.post_snapshots if which == "post" else rec.pre_snapshots
    return seq[i] if 0 <= i < len(seq) else None


def short_rows(s):
    # Every resting row, both sides, as sorted [price_int, qty_int, oid].
    if s is None:
        return None
    return {side_name(sd): sorted([int(x[0]), int(x[1]), int(x[2])] for x in s.resting(sd)) for sd in (BUY, SELL)}


def stamps(s, oid, side):
    # The stored [sec, nsec] of every book entry of `oid` on `side` (None when the snapshot is missing).
    if s is None:
        return None
    return [[int(x[4]), int(x[5])] for x in s.resting(side) if int(x[2]) == int(oid)]


def single(st):
    return st[0] if st is not None and len(st) == 1 else None


def stamp_window(rec, e):
    # (T1): the owner's stamps in every book from its first rest to just before the take.
    win = [(f"post[{k}]", snap(rec, "post", k)) for k in range(e["res_entry"], TAKE_SUBMISSION)]
    win.append((f"pre[{TAKE_SUBMISSION}]", snap(rec, "pre", TAKE_SUBMISSION)))
    return [[name, stamps(s, e["res_oid"], e["res_side"])] for name, s in win]


def window_ok(win):
    st = [single(x[1]) for x in win]
    return bool(win) and all(v is not None for v in st) and all(v == st[0] for v in st)


def observe(rec, e):
    # What the engine did on one run, in the expectation's terms (None entries for missing snapshots).
    fills = sorted(rec.fills, key=lambda f: (int(f.submission), int(f.row_index)))
    seq = [[int(f.passive_oid), int(f.aggressor_oid), int(f.price_int), int(f.qty_int), int(f.submission)]
           for f in fills]
    p1 = snap(rec, "post", PARTIAL_SUBMISSION)
    pt = snap(rec, "pre", TAKE_SUBMISSION)
    fin = rec.final_snapshot
    res, rside = e["res_oid"], e["res_side"]
    final_rest = {o: int(sum(int(x[1]) for sd in (BUY, SELL) for x in fin.resting(sd) if int(x[2]) == o))
                  for o in e["oids"]}
    filled = {o: int(rec.filled.get(o, 0)) for o in e["oids"]}
    return {
        "accepted": sorted(int(o) for o in rec.accepted), "rejected": sorted(int(r.oid) for r in rec.rejections),
        "seq": seq,
        "partial_fill_qty": int(sum(f[3] for f in seq if f[4] == PARTIAL_SUBMISSION)),
        "res_qty": None if p1 is None else int(p1.visible_qty(res, rside)),
        "res_prices": None if p1 is None else sorted({int(x[0]) for x in p1.resting(rside) if int(x[2]) == res}),
        "book_after_partial": short_rows(p1),
        "window": stamp_window(rec, e),
        "res_stamp_1": single(stamps(p1, res, rside)),
        "res_stamp_take": stamps(pt, res, rside),
        "int_stamp_take": stamps(pt, e["interloper_oid"], e["interloper_side"]),
        "take_passives": [f[0] for f in seq if f[4] == TAKE_SUBMISSION],
        "interloper_filled": filled.get(e["interloper_oid"], 0),
        "book_final": short_rows(fin),
        "conservation": {o: filled[o] + final_rest[o] for o in e["oids"]},
        "filled": filled,
    }


def check_run(label, seed, rec, e, ctrl_stamp, kind, report=None):
    # (A), (Q), (F), (R), (B1), (T1)-(T3), (O), (B), (I2) of section 5. Returns (observation, all passed).
    o = observe(rec, e)
    ok = cert.check_equal(f"{label}: (A) accepted order ids / rejected order ids",
                          [o["accepted"], o["rejected"]], [sorted(e["oids"]), []], seed=seed, unit="other",
                          rule=R_ACC)
    ok &= cert.check_equal(f"{label}: (Q) counterparty sequence [passive, aggressor, price_int, qty_int, "
                           f"submission]", o["seq"], e["fills"], seed=seed, unit="other", rule=R_SEQ)
    ok &= cert.check_equal(f"{label}: (F) quantity traded at the partial fill (submission 1)",
                           o["partial_fill_qty"], e["partial_fill_qty"], seed=seed, unit="qty_int", rule=R_PART)
    ok &= cert.check_equal(f"{label}: (R) residual quantity of oid {e['res_oid']} after the partial fill",
                           o["res_qty"], e["res_qty"], seed=seed, unit="qty_int", rule=R_PART)
    ok &= cert.check_equal(f"{label}: (R) residual resting price(s) of oid {e['res_oid']} after the partial fill "
                           f"(its own limit)", o["res_prices"], e["res_prices"], seed=seed, unit="price_int",
                           rule=R_PART + "; §6.2 item 3")
    ok &= cert.check_equal(f"{label}: (B1) book after the partial fill [price_int, qty_int, oid]",
                           o["book_after_partial"], e["book_after_partial"], seed=seed, unit="other", rule=R_BOOK)
    ok &= cert.check(f"{label}: (T1) stored [sec, nsec] of oid {e['res_oid']} unchanged from its first rest to the "
                     f"take", window_ok(o["window"]), seed=seed,
                     expected="one identical [sec, nsec] in every snapshot listed", observed=o["window"],
                     unit="other", rule=R_TS)
    if kind != "resting":
        t2 = o["res_stamp_1"] is not None and ctrl_stamp is not None and o["res_stamp_1"] == ctrl_stamp
        ok &= cert.check(f"{label}: (T2) residual's stored [sec, nsec] after the partial fill == the aggressor's "
                         f"entry stamp in the no_cross_control", t2, seed=seed, expected=ctrl_stamp,
                         observed=o["res_stamp_1"], unit="other", rule=R_TS_ENTRY)
    rs_, is_ = single(o["res_stamp_take"]), single(o["int_stamp_take"])
    ok &= cert.check(f"{label}: (T3) just before the take, residual's [sec, nsec] < interloper's",
                     rs_ is not None and is_ is not None and tuple(rs_) < tuple(is_), seed=seed,
                     expected="residual strictly earlier (lexicographic), one entry each",
                     observed={"residual": o["res_stamp_take"], "interloper": o["int_stamp_take"]}, unit="other",
                     rule=R_TS_ORDER)
    ok &= cert.check_equal(f"{label}: (O) take's passive oids / interloper's filled qty_int",
                           [o["take_passives"], o["interloper_filled"]], [e["take_passives"], 0], seed=seed,
                           unit="other", rule=R_SEQ)
    ok &= cert.check_equal(f"{label}: (B) book at the end [price_int, qty_int, oid]", o["book_final"],
                           e["book_final"], seed=seed, unit="other", rule=R_BOOK)
    ok &= cert.check_equal(f"{label}: (I2) filled + resting per oid == declared qty_int", o["conservation"],
                           e["declared"], seed=seed, unit="qty_int", rule=R_I2)
    if report is not None:
        st = report.statuses().get("I2")
        ok &= cert.check_equal(f"{label}: (I2) battery I2 status", st, "pass", seed=seed, unit="other", rule=R_I2)
    return o, bool(ok)


def check_control(label, seed, rec, tape):
    # The no_cross_control: (A), (Q), (B1), (B). Returns (a's stamp after submission 1, all passed).
    w = walk(tape)
    oids = [int(x[C_OID]) for x in real_rows(tape)]
    fills = sorted(rec.fills, key=lambda f: (int(f.submission), int(f.row_index)))
    seq = [[int(f.passive_oid), int(f.aggressor_oid), int(f.price_int), int(f.qty_int), int(f.submission)]
           for f in fills]
    ok = cert.check_equal(f"{label}: (A) accepted order ids / rejected order ids",
                          [sorted(int(x) for x in rec.accepted), sorted(int(r.oid) for r in rec.rejections)],
                          [sorted(oids), []], seed=seed, unit="other", rule=R_ACC)
    ok &= cert.check_equal(f"{label}: (Q) counterparty sequence", seq, w["fills"], seed=seed, unit="other",
                           rule=R_SEQ)
    ok &= cert.check_equal(f"{label}: (B1) book after submission 1 (a rests whole)",
                           short_rows(snap(rec, "post", PARTIAL_SUBMISSION)), w["books"][PARTIAL_SUBMISSION],
                           seed=seed, unit="other", rule=R_BOOK)
    ok &= cert.check_equal(f"{label}: (B) book at the end", short_rows(rec.final_snapshot), w["books"][-1],
                           seed=seed, unit="other", rule=R_BOOK)
    a = real_rows(tape)[1]
    st = stamps(snap(rec, "post", PARTIAL_SUBMISSION), int(a[C_OID]), int(a[C_SIDE]))
    ok &= cert.check(f"{label}: the aggressor holds exactly one book entry after submission 1", single(st) is not None,
                     seed=seed, expected="one [sec, nsec]", observed=st, unit="other", rule=R_TS_ENTRY)
    return single(st), bool(ok)


def battery_check(label, seed, rec):
    rep = run_battery(rec, expects_trade=True)
    return cert.check(f"{label}: battery (RB, I1-I9) passes", rep.passed and rep.valid, seed=seed,
                      expected="pass / not_applicable", observed=rep.statuses(), unit="other", rule="battery")


EAGER_DONE = {}


def eager_compare(label, seed, tape, rec, tag):
    # Run `tape` eager and compare with its jitted record; recorded as one eager_jit assertion.
    rec_e = guarded(f"{label} (eager)", seed, run_tape, tape, jit=False)
    if rec is None or rec_e is None:
        diff = ["jitted run missing" if rec is None else "eager run missing"]
    else:
        diff = list(compare_records(rec_e, rec))
    EAGER_DONE.setdefault(tag, seed)
    cert.set_eager_jit(seed, diff == [], detail=f"{label}: " + (", ".join(diff) if diff else "identical"))
    return diff == []


def margin_breaches(tape, bounds):
    # The capacity-validity precondition (five ceilings), checked from the tape before it is run.
    lim = {"resting_per_side": ENVELOPE.max_resting_per_side, "trades": ENVELOPE.max_trades,
           "hops": ENVELOPE.max_hops, "concurrent_icebergs": ENVELOPE.max_concurrent_icebergs}
    bad = [f"{k} bound {bounds.get(k)} > {m}" for k, m in lim.items() if int(bounds.get(k, 0)) > m]
    if max_slices(tape) > SLICE_MARGIN:
        bad.append(f"ceil(qty/peak) {max_slices(tape)} > {SLICE_MARGIN}")
    return bad


cert.check("section 5: closed form agrees with the walk (W) on every run of the first seeds", CF_AGREES,
           expected=True, observed=CF_AGREES, unit="bool", rule=R_SEQ)
wired = callable(run_tape)
cert.check("section 6 wired (run_tape defined)", wired, expected=True, observed=wired, unit="bool",
           detail="" if wired else "section 6 is still the K2 stub", rule=R_ACC)

PER_RUN, DATA_BAD, CLOSED_BAD, SEP_BAD, DERIVED_BAD = [], defaultdict(dict), [], [], []
CLASS_COUNT, COVER, OBS = Counter(), defaultdict(Counter), Counter()
PEAK = {"hop_bound": 0, "max_slices": 0}


def run_derived(key, seed, kind, draw, lbl, side, ctrl_stamp=None):
    # One derived run (same_second or no_cross_control): margin, closed form, separation, run, checks, eager.
    t = DERIVED[(key, seed, kind)]
    bad = margin_breaches(t, synth.tape_capacity_bounds(t))
    if bad or any(not 0 <= int(x[C_PRICE]) <= PRICE_INT_MAX for x in real_rows(t)):
        DERIVED_BAD.append(f"{lbl} {kind}: " + ("; ".join(bad) if bad else "outside §2.3"))
        return None
    if not closed_form_agrees(key, kind, draw, t):
        CLOSED_BAD.append(f"{key} seed {seed} {kind}")
    if kind != "no_cross_control" and not separates(t):
        SEP_BAD.append(f"{key} seed {seed} {kind}")
    l2 = f"{lbl} {kind}"
    rec = guarded(f"{l2} (jit)", seed, run_tape, t, jit=True)
    if rec is None:
        return None
    battery_check(l2, seed, rec)
    if kind == "no_cross_control":
        out, ok = check_control(l2, seed, rec, t)
    else:
        o, ok = check_run(l2, seed, rec, expectation(t), ctrl_stamp, "aggressor" if key == AGGRESSOR else "resting")
        PER_RUN.append({"key": key, "seed": seed, "run": kind, "ok": ok})
        out = o
    if (key, kind, side) not in EAGER_DONE:
        eager_compare(l2, seed, t, rec, (key, kind, side))
    return out


def run_construction(key):
    first, n = FIRST_SEED[key], N_DRAWS[key]
    valid, seed = 0, first
    while valid < n and seed < first + 2 * n:
        draw = DRAWN.get((key, seed)) or prepare(key, seed)
        margin = margin_breaches(draw.tape, dict(draw.capacity_bounds))
        if margin:
            cert.discard(seed, f"{key}: pre-run capacity margin: " + "; ".join(margin))
            seed += 1
            continue
        bad = precondition(key, draw)
        e = expectation(draw.tape)
        if bad or e is None:
            DATA_BAD[key][seed] = bad or ["expectation undefined"]
            cert.check(f"{key} seed {seed}: draw is as section 4 describes", False, seed=seed, expected=[],
                       observed=DATA_BAD[key][seed], unit="other", rule=R_DATA)
            seed += 1
            continue
        lbl = f"{key} seed {seed}"
        rec = guarded(f"{lbl} (jit)", seed, run_tape, draw.tape, jit=True)
        report = run_battery(rec, expects_trade=True) if rec is not None else None
        if report is not None and not cert.add_battery(report, seed=seed):
            seed += 1   # I8 invalid: discarded, re-drawn with the next seed
            continue
        valid += 1
        cert.record_case(seed, draw.tape, {**draw.params, "construction": key})
        PEAK["hop_bound"] = max(PEAK["hop_bound"], int(draw.hop_bound))
        PEAK["max_slices"] = max(PEAK["max_slices"], max_slices(draw.tape))
        side = side_name(draw.params["rest_side"])
        CLASS_COUNT[(key, side)] += 1
        COVER[key]["q_i > residual" if int(real_rows(draw.tape)[2][C_QTY]) > e["res_qty"] else "q_i <= residual"] += 1
        if not closed_form_agrees(key, "draw", draw, draw.tape):
            CLOSED_BAD.append(f"{key} seed {seed} draw")
        if not separates(draw.tape):
            SEP_BAD.append(f"{key} seed {seed} draw")

        ctrl_stamp = None
        if key == AGGRESSOR:
            ctrl_stamp = run_derived(key, seed, "no_cross_control", draw, lbl, side)
        if rec is not None:
            cert.record_capacity({**rec.observed_capacity, "hops": None})   # hops: unobserved by the engine
            o, ok = check_run(lbl, seed, rec, e, ctrl_stamp, "aggressor" if key == AGGRESSOR else "resting",
                              report=report)
            PER_RUN.append({"key": key, "seed": seed, "run": "draw", "ok": ok})
            res_row = real_rows(draw.tape)[e["res_entry"]]
            if o["res_stamp_1"] is not None:
                OBS[(key, "stored sec == residual row's tape ts", o["res_stamp_1"][0] == int(res_row[C_TS]))] += 1
                OBS[(key, "residual nsec", o["res_stamp_1"][1])] += 1
            if valid <= EAGER_DRAWS or (key, "draw", side) not in EAGER_DONE:
                eager_compare(lbl, seed, draw.tape, rec, (key, "draw", side))
        run_derived(key, seed, "same_second", draw, lbl, side, ctrl_stamp)
        seed += 1
    return valid


VALID = {}
if wired:
    for _key in CONSTRUCTIONS:
        VALID[_key] = run_construction(_key)

# ---- aggregate assertions (every list in full) ----------------------------------------------------------
for _key in CONSTRUCTIONS:
    _bad = DATA_BAD.get(_key, {})
    cert.check(f"dataset: every {_key} draw is as section 4 describes (precondition)", wired and not _bad,
               expected={}, observed={str(k): v for k, v in _bad.items()}, unit="other",
               detail=f"{len(_bad)} draws depart" if _bad else f"{VALID.get(_key, 0)} valid draws", rule=R_DATA)
cert.check("section 5: closed form agrees with the walk (W) on every run", wired and not CLOSED_BAD,
           expected=[], observed=CLOSED_BAD, unit="other", rule=R_SEQ)
cert.check("dataset: every derived tape is within §2.3's bounds and the capacity margin", wired and not DERIVED_BAD,
           expected=[], observed=DERIVED_BAD, unit="other", rule="§2.3; capacity-validity precondition")
cert.check("dataset separates the rule from the re-stamp model on every draw and same_second run",
           wired and not SEP_BAD, expected=[], observed=SEP_BAD, unit="other", rule=R_TS)
for _key in CONSTRUCTIONS:
    _want = {"resting BUY": ">= 1", "resting SELL": ">= 1", "q_i > residual": ">= 1", "q_i <= residual": ">= 1"}
    _got = {"resting BUY": CLASS_COUNT[(_key, "BUY")], "resting SELL": CLASS_COUNT[(_key, "SELL")],
            "q_i > residual": COVER[_key]["q_i > residual"], "q_i <= residual": COVER[_key]["q_i <= residual"]}
    _missing = [k for k, v in _got.items() if v == 0]
    cert.check(f"dataset: {_key} covers both resting sides and an interloper larger / not larger than the residual",
               wired and not _missing, expected=_want, observed=_got, unit="count",
               detail=f"missing: {_missing}" if _missing else "", rule=R_DATA)

# ---- eager: every required class must have had one comparison (absence = failure) -----------------------
REQUIRED_EAGER = [(k, kind, sd) for k in CONSTRUCTIONS for kind in RUN_KINDS[k] for sd in ("BUY", "SELL")]
for _tag in REQUIRED_EAGER:
    if _tag not in EAGER_DONE:
        cert.check(f"eager == jitted for class {_tag}", False, kind="eager_jit", expected=True, observed=None,
                   unit="bool", detail=f"no eager comparison was run for required class {_tag}")

# ---- reports -----------------------------------------------------------------------------------------------
_fail_runs = [r for r in PER_RUN if not r["ok"]]
display(Markdown(f"**Per-run summary.** {len(PER_RUN)} draw / same_second runs checked, {len(_fail_runs)} with a "
                 f"failed assertion" + (": " + ", ".join(f"{r['key']} seed {r['seed']} {r['run']}"
                                                          for r in _fail_runs) if _fail_runs else ".")))
print(f"valid draws per construction: {VALID}; eager comparisons run for classes: {sorted(EAGER_DONE)}")
print(f"analytic peaks over valid draws: hop bound {PEAK['hop_bound']}, max ceil(qty/peak) {PEAK['max_slices']}")
print("observed (reported, not asserted; finding F3):")
for _k, _n in sorted(OBS.items(), key=lambda kv: tuple(str(x) for x in kv[0])):
    print(f"    {_k}: {_n}")
display(Markdown(cert.table()))


## 8. Verdict

Emit the results JSON (validated against `results.schema.json`) and print exactly one word:
`PASS` or `NOT CERTIFIED`. On `NOT CERTIFIED` the smallest failing case is frozen as a fixture.

In [ ]:
RESULTS_PATH = cert.emit()
print(cert.verdict_line())
print(f"results -> {RESULTS_PATH}")

In [ ]:
# Section 9 ("Why this certification failed") is rendered here only on NOT CERTIFIED; it is
# persisted as the last markdown cell at K6-F by cert_support.append_failure_section.
_section9 = cert.failure_section_markdown()
if _section9:
    display(Markdown(_section9))